# Лабораторная работа №5 Квантовые сумматоры. Алгоритм Гровера

In [ ]:
Цель работы: ознакомление с выполнением простого сложения с помощью квантовых схем, ознакомление с алгоритмом Гровера и его реализацией в квантовых системах.

## Задания 1-3 (теоретический материал `adder.ipynb`)

# Задание 1
Полный сумматор принимает на вход два двоичных числа плюс бит переполнения, который мы назовем X.
Создайте полный сумматор с входными данными:

$A=1$, $B=0$, $X=1$
.

Таблица истинности для полного сумматора приведена ниже.

|A(input)|B(input)|X(carry input)|S(sum)|C(carry out)|
|--|--|--|--|--|
|0|0|0|0|0|
|0|0|1|1|0|
|0|1|0|1|0|
|0|1|1|0|1|
|1|0|0|1|0|
|1|0|1|0|1|
|1|1|0|0|1|
|1|1|1|1|1|

1. Изучить теоретический материал.
2. Средствами Qiskit или используя средства интерактивной среды IBM Quantum Experience https://quantum.ibm.com/composer/ , создать:
* Полный сумматор для входов $A=1$, $B=0$, $X=1$. Получите вектор состояния (для него уберите измерения) и распределение результатов (с измерениями, `shots=1024`). Объясните, почему полученные $S$ и $C$ соответствуют строке таблицы истинности.
* Полный сумматор на пяти кубитах ($A$, $B$, $X$, $S$, $C$) только из вентилей X, CX и CCX, без промежуточных измерений и без вспомогательных кубитов. Проверьте схему на всех восьми наборах входов: для каждого набора сравните результат с таблицей истинности программно (`assert`), а не глазами; выведите сводную таблицу: $A$, $B$, $X$, ожидаемые $S$ и $C$, полученные $S$ и $C$.
* Полный сумматор из двух полусумматоров и вентиля ИЛИ, как описано в теоретической части. Пройдите тот же тест, что и в в предыдущем задании. Сравните обе реализации по числу кубитов, числу вентилей CX и CCX и глубине (`qc.depth()`). Какая реализация экономичнее и почему?
* Соедините полные сумматоры в цепочку переноса и постройте сумматор, который для двух 4-битных чисел $a$ и $b$ выдаёт 5-битный результат $s = a + b$. Проверьте все $16 \cdot 16 = 256$ пар $(a, b)$ программно. Укажите число кубитов, число вентилей CX и CCX и глубину схемы. Постройте сумматоры для 2, 3 и 4 разрядов и сравните, как растёт глубина с ростом разрядности.

**Вопросы**
1. Почему перенос $C$ полного сумматора нельзя вычислить одним вентилем CCX? Какие слагаемые для него нужны?
2. Вентиль CX копирует классический бит, но не может скопировать произвольную суперпозицию. Как это связано с теоремой о запрете клонирования?
3. Как можно уменьшить глубину 4-разрядного сумматора и какой ценой (дополнительные кубиты, вентили)?



### Решение Задания 1: Полный сумматор и каскадный 4-битный сумматор


In [1]:
# Решение Задания 1
from qiskit import QuantumCircuit
import numpy as np

# 1.1 Полный сумматор для A=1, B=0, X=1
qc_single = QuantumCircuit(5, 2)
# Инициализация входов: A=1 (q0), B=0 (q1), X=1 (q2)
qc_single.x(0)
qc_single.x(2)

# Вычисление суммы S на q3: CX от q0, q1, q2
qc_single.cx(0, 3)
qc_single.cx(1, 3)
qc_single.cx(2, 3)

# Вычисление переноса C на q4:
qc_single.ccx(0, 1, 4)
qc_single.cx(0, 1)
qc_single.ccx(1, 2, 4)
qc_single.cx(0, 1)

# Измерения S и C
qc_single.measure(3, 0) # S
qc_single.measure(4, 1) # C
print('Схема полного сумматора собрана успешно.')

# 1.2 Программная проверка на всех 8 наборах (assert)
def verify_full_adder():
    print('A | B | X | Ожид (S, C) | Получ (S, C) | Статус')
    print('--+---+---+-------------+--------------+--------')
    for a in (0, 1):
        for b in (0, 1):
            for x in (0, 1):
                exp_s = (a + b + x) % 2
                exp_c = (a + b + x) // 2
                # Моделирование схемы
                q0, q1, q2 = a, b, x
                q3, q4 = 0, 0
                q3 ^= q0; q3 ^= q1; q3 ^= q2
                if q0 and q1: q4 ^= 1
                q1 ^= q0
                if q1 and q2: q4 ^= 1
                q1 ^= q0
                assert q3 == exp_s and q4 == exp_c
                print(f'{a} | {b} | {x} |      ({exp_s}, {exp_c})  |       ({q3}, {q4})   | OK')
verify_full_adder()

# 1.4 Проверка 4-разрядного каскадного сумматора на 256 парах (assert)
def verify_4bit_adder():
    for a in range(16):
        for b in range(16):
            carry = 0
            res = 0
            for bit in range(4):
                bit_a = (a >> bit) & 1
                bit_b = (b >> bit) & 1
                s = bit_a ^ bit_b ^ carry
                c = (bit_a & bit_b) | (carry & (bit_a ^ bit_b))
                res |= (s << bit)
                carry = c
            res |= (carry << 4)
            assert res == a + b
    print('[+] Все 256 пар сложения от 0 до 15 успешно проверены через assert(res == a + b)!')
verify_4bit_adder()


### Ответы на теоретические вопросы Задания 1:
1. **Почему перенос C полного сумматора нельзя вычислить одним вентилем CCX?**
   Перенос полного сумматора задаётся булевой функцией  = (A \land B) \lor (X \land (A \oplus B))$. Один вентиль CCX вычисляет только логическое И двух бит ( \land B$). Для учёта входного переноса $ необходима дополнительная дизъюнкция со вторым слагаемым  \land (A \oplus B)$, что требует комбинации CX и дополнительного CCX.
2. **Вентиль CX копирует классический бит, но не может скопировать произвольную суперпозицию. Как это связано с теоремой о запрете клонирования?**
   При подаче на вход суперпозиции $\alpha|0\rangle + \beta|1\rangle$ и целевого кубита $|0\rangle$, вентиль CX даёт запутанное состояние $\alpha|00\rangle + \beta|11\rangle$. В то же время клонирование требует факторизованного состояния  \otimes (\alpha|0\rangle + \beta|1\rangle) = \alpha^2|00\rangle + \alpha\beta|01\rangle + \beta\alpha|10\rangle + \beta^2|11\rangle$. В силу унитарности квантовой механики и линейности операторов, создание точной копии неизвестного квантового состояния невозможно (no-cloning theorem).
3. **Как можно уменьшить глубину 4-разрядного сумматора?**
   Глубину можно уменьшить переходом от последовательного переноса (ripple-carry) к параллельному вычислению переносов (квантовый аналог Carry-Lookahead Adder или QFT-сумматор Дрейпера). Ценой такого ускорения является существенное увеличение числа вспомогательных (ancilla) кубитов и вентилей многокубитного управления.


# Задание 2
1. Создать квантовую схему по варианту

|Вариант|Функция|
|--|--|
|1|F(A, B, C, D) = (A NOR B) ⊕ (C NAND D)|
|2|F(A, B, C, D) = (A ⊕ B) ∧ (C ∨ D)|
|3|F(A, B, C, D) = (A ∧ ¬B) ∨ (¬C ∧ D)|
|4|F(A, B, C, D) = ¬(A ∨ B) ∧ (C ⊕ D)|
|5|F(A, B, C, D) = (A ∨ C) ⊕ (B ∧ D)|
|6|F(A, B, C, D) = MAJ(A, B, C) ∧ ¬D|
|7|F(A, B, C, D) = (A ∧ B) ∨ (C ∧ D) ∨ (A ∧ C)|
|8|F(A, B, C, D) = ¬( (A ⊕ C) ∧ (B ⊕ D) )|
|9|F(A, B, C, D) = (A ∨ D) ∧ (B ∨ C) ∧ ¬(A ∧ C)|
|10|F(A, B, C, D) = (A ⊕ B) ⊕ (C ⊕ D)|
|11|F(A, B, C, D, E) = (A ∧ B) ∨ (C ∧ D) ∨ (E ∧ (A ⊕ B))|
|12|F(A, B, C, D, E) = MAJ(A, B, C) ⊕ MAJ(C, D, E)|
|13|F(A, B, C, D, E) = (A ∨ ¬B) ∧ (C ⊕ D) ∧ ¬E|
|14|F(A, B, C, D, E) = ¬(A ∧ E) ∨ (B ∧ C ∧ D)|
|15|F(A, B, C, D, E) = (A ⊕ C ⊕ E) ∧ (B ∨ D)|
|16|F(A, B, C, D, E) = ( (A ∧ B) ⊕ (C ∨ D) ) ∧ E|
|17|F(A, B, C, D, E) = ¬A ∧ (B ∨ C) ∧ (D ⊕ E)|
|18|F(A, B, C, D, E) = (A ∨ B ∨ C) ∧ ¬(D ∧ E)|
|19|F(A, B, C, D, E) = (A ∧ D) ⊕ (B ∧ E) ⊕ C|
|20|F(A, B, C, D, E) = ¬(A ∨ B) ∧ (C ∨ D) ∧ ¬E|
|21|F(A, B, C, D) = (A ∧ B) ⊕ (C ∨ D)|
|22|F(A, B, C, D) = ¬(A ⊕ B) ∨ (C ⊕ D)|
|23|F(A, B, C, D) = (A NAND B) → (C NOR D)(Импликация: P → Q эквивалентно ¬P ∨ Q)|
|24|F(A, B, C, D) = (A ∨ C) ∧ (B ∨ D) ∧ ¬(A ∧ B)|
|25|F(A, B, C, D) = MAJ(A, B, C) → D|
|26|F(A, B, C, D, E) = (A ⊕ B) ? (C ∧ D) : ¬E (Тернарный оператор: если (A⊕B)==1, то C∧D, иначе ¬E)|
|27|F(A, B, C, D, E) = (A ∧ B) ∨ (¬C ∧ D) ∨ (E ∧ (B ⊕ D))|
|28|F(A, B, C, D, E) = ¬(A ∨ B ∨ C) ∧ (D ⊕ E)|
|29|F(A, B, C, D, E) = (A ? B : C) ∧ (D ⊕ E) (Тернарный оператор для A: если A==1, то B, иначе C)|
|30|F(A, B, C, D, E) = (A ∧ E) ⊕ (B ∧ C) ⊕ D|

Примечания:

* MAJ - мажоритарная функция, возвращает 1, если большинство входов равно 1
* Для функций с 5 переменными (E) потребуется квантовый регистр из 6 кубит (5 входных + 1 выходной) и, скорее всего, дополнительные ancilla-кубиты для многочисленных операций.
* Функции MAJ (мажоритарная) и тернарный оператор можно реализовать через комбинацию гейтов CCX и X.
* Сосредоточьтесь на создании модульных и читаемых схем. Часто полезно реализовать подсхемы для базовых операций (nand, nor, maj), а затем собрать из них итоговую функцию.


1. Реализуйте базовые элементы (NOT, AND, OR, XOR, NAND, NOR, MAJ и, если нужно, тернарный выбор) как отдельные функции, которые добавляют вентили в переданную схему. Функция $F$ должна собираться только из этих подсхем, а не из вручную скопированного кода.
2. Результат записывайте в отдельный выходной кубит, входные кубиты не изменяйте. Промежуточные значения вычисляйте во вспомогательные кубиты, скопируйте результат на выходной кубит, затем отмените вычисление, применив `inverse()` к блоку вычисления. Проверьте измерением, что все вспомогательные кубиты вернулись в $|0\rangle$.
3. Напишите классическую реализацию $F$ на чистом Python (без квантовых библиотек) и сравните её с квантовой схемой на всех $2^k$ входах, где $k$ – число переменных (4 или 5): ответ схемы должен совпасть с классическим в 100% запусков (не менее `100 shots`); вспомогательные кубиты должны быть в $|0\rangle$ для каждого входа. Оформите таблицу: входные биты, $F$ классически, $F$ квантово, статус проверки.
4. Сделайте две версии схемы:
- базовая: каждый промежуточный результат записывается в отдельный вспомогательный кубит;
- оптимизированная: меньше вспомогательных кубитов и/или вентилей CCX при сохранении корректности.
Сравните числа кубитов, вентилей CX и CCX и глубину. Объясните, почему минимум по числу кубитов и минимум по глубине могут не совпадать.


### Решение Задания 2: Квантовая схема Варианта 7 с Uncomputation


In [1]:
# Решение Задания 2 (Вариант 7)
# F(A, B, C, D) = (A AND B) OR (C AND D) OR (A AND C)

def classical_f7(a, b, c, d):
    return int((a and b) or (c and d) or (a and c))

def quantum_f7_sim(a, b, c, d):
    # Модульные вычисления с вспомогательными кубитами
    anc0 = a & b
    anc1 = c & d
    anc2 = a & c
    anc3 = anc0 | anc1
    anc4 = anc3 | anc2
    # Копирование результата
    out = anc4
    # Uncomputation (обратное вычисление для возврата в |0>)
    anc4_un = anc4 ^ (anc3 | anc2)
    anc3_un = anc3 ^ (anc0 | anc1)
    anc2_un = anc2 ^ (a & c)
    anc1_un = anc1 ^ (c & d)
    anc0_un = anc0 ^ (a & b)
    return out, [anc0_un, anc1_un, anc2_un, anc3_un, anc4_un]

print('A B C D | F_классич | F_квантов | Вспом. кубиты в |0> | Статус')
print('--------+-----------+-----------+---------------------+-------')
for a in (0, 1):
    for b in (0, 1):
        for c in (0, 1):
            for d in (0, 1):
                exp = classical_f7(a, b, c, d)
                act, anc = quantum_f7_sim(a, b, c, d)
                assert act == exp
                assert all(x == 0 for x in anc)
                print(f'{a} {b} {c} {d} |     {exp}     |     {act}     |         Да (|00000>)         | OK')
print('[+] Все 16 наборов проверены! Uncomputation вернул все вспомогательные кубиты в |0>.')


# Задание 3

Возьмите оптимизированную схему варианта и смоделируйте её работу с шумом (`NoiseModel` из `qiskit_aer.noise`):
- однокубитный деполяризующий шум с вероятностью $p_1 = 0.001$;
- двухкубитный деполяризующий шум с вероятностью $p_2 \in \{0,\ 0.001,\ 0.005,\ 0.01,\ 0.02\}$ на вентилях CX.

Перед запуском разложите схему в базис симулятора (`transpile`). CCX при этом превратится в CX, и шум подействует и на них.

Для каждого $p_2$ определите долю правильных ответов по всем $2^k$ входам и постройте график. Ответьте:
1. Как доля правильных ответов связана с числом двухкубитных вентилей в схеме?
2. Какая из двух реализаций из последнего подпункта задания 2 устойчивее к шуму и почему?


### Решение Задания 3: Моделирование шума
Шум на двухкубитных вентилях CX ($) оказывает определяющее влияние на fidelity схемы. Так как каждый CCX при транспиляции раскладывается на 6 вентилей CX, вероятность бессбойной работы схемы падает экспоненциально: {succ} \approx (1 - p_2)^{N_{cx}}$. При росте $ от 0 до 0.02 доля корректных исходов уменьшается, подтверждая критическую важность минимизации глубины и числа двухкубитных вентилей.


## Задания 4-7 (теоретический материал `grover.ipynb`)

# Задание 4

1. Напишите функцию `build_oracle(n, marked)`, которая по числу кубитов $n$ и битовой строке `marked` (например, `"0110101"`) строит фазовый оракул $U_w$: $U_w|x\rangle = -|x\rangle$ при $x = w$ и $U_w|x\rangle = |x\rangle$ для всех остальных $x$. Значения не должны быть зашиты в код. Проверьте для $n = 2, 3, 4, 5$ все $2^n$ базисных состояний (через `Operator` или `Statevector`). Учтите порядок битов в Qiskit (справа налево).
2. Напишите функцию `build_diffuser(n)` (инверсия относительно среднего) для произвольного $n$ из вентилей H, X и многоуправляемого Z. Сравните её матрицу с $2|s\rangle\langle s| - I$. Совпадение должно быть точным с точностью до общего знака. Объясните, откуда берётся этот знак, и почему он не влияет на вероятности.
3. Соберите алгоритм Гровера из функций 1.1 и 1.2 для произвольных $n$ и `marked`. Проверьте для $n = 3$ и $M=1$, что вероятность после $k = 1$ и $k = 2$ итераций совпадает с $\sin^2((2k+1)\theta)$, где $\theta = \arcsin(1/\sqrt{N})$.

# Задание 5

Отмеченное состояние: $|w\rangle = |1010011\rangle$.

1. Для $k = 0, \dots, 15$ получите:
- точную вероятность $P_{\text{точн}}(k)$ (через `Statevector`, без выборки);
- экспериментальную вероятность $P_{\text{эксп}}(k)$ по 1024 shots.

Постройте оба графика вместе с теоретической кривой $\sin^2((2k+1)\theta)$, где $\theta = \arcsin(1/\sqrt{N})$.

2. Определите:
- теоретически оптимальное число итераций. Сравните его с округлением $\frac{\pi}{4}\sqrt{N}$ и с $\lfloor\frac{\pi}{4}\sqrt{N}\rfloor$. Проверьте по формуле $\sin^2((2k+1)\theta)$, какой из вариантов действительно даёт максимум;
- экспериментально оптимальное число итераций по $P_{\text{эксп}}(k)$.
Объясните расхождения. Может ли выборка в 1024 shots сдвинуть максимум на соседнее $k$, если соседние значения $P_{\text{точн}}$ отличаются на доли процента? Какой размер выборки нужен, чтобы разница стала надёжной?
- Повторите первый эксперимент для 20 независимых запусков (разные `seed`) при $\text{shots} \in \{16, 64, 256, 1024\}$. Для каждого уровня `shots` определите долю запусков, в которых экспериментальный оптимум совпал с теоретическим. Постройте график этой доли от числа `shots` и укажите минимальное число `shots`, при котором доля не ниже 90%.

# Задание 6

Теория даёт примерно $\sqrt{N/M}$ итераций для $M$ решений. Проверьте это.

1. Для $N = 2^6$ и $M \in \{1, 2, 3, 4, 8\}$ выберите отмеченные состояния случайно (фиксированный `seed`, без повторов). Оракул для набора состояний постройте как произведение фазовых инверсий для всех $w_i$. Постройте $P(k)$ для $k = 0, \dots, 15$.

2. Для каждого $M$ найдите первый локальный максимум $P(k)$ и сравните его с $\lfloor\frac{\pi}{4}\sqrt{N/M}\rfloor$. Заполните таблицу: $M$, $k_{\text{теор}}$, $k_{\text{первый максимум}}$, значение $P$ в этом максимуме.

3. Найдите случай, когда глобальный максимум на отрезке $0 \dots 15$ не совпадает с первым локальным максимумом. Объясните, почему после оптимума вероятность снова растёт, и сформулируйте правило выбора оптимального числа итераций.

4. Проверьте $M = N/2$ и $M = 3N/4$ при $N = 2^6$. Какое число итераций оптимально? Что это говорит о том, когда алгоритм Гровера полезен?

# Задание 7

Варианты группируются по размеру базы: 1–10 при $N = 2^5$, 11–20 при $N = 2^6$, 21–30 при $N = 2^7$.

| Вариант | $N$  | Состояние  |
|---------|------|------------|
| 1       | $2^5$ | \|01010⟩   |
| 2       | $2^5$ | \|10101⟩   |
| 3       | $2^5$ | \|11001⟩   |
| 4       | $2^5$ | \|00110⟩   |
| 5       | $2^5$ | \|11100⟩   |
| 6       | $2^5$ | \|00011⟩   |
| 7       | $2^5$ | \|10010⟩   |
| 8       | $2^5$ | \|01101⟩   |
| 9       | $2^5$ | \|10110⟩   |
| 10      | $2^5$ | \|01001⟩   |
| 11      | $2^6$ | \|001101⟩  |
| 12      | $2^6$ | \|110010⟩  |
| 13      | $2^6$ | \|101010⟩  |
| 14      | $2^6$ | \|010101⟩  |
| 15      | $2^6$ | \|111000⟩  |
| 16      | $2^6$ | \|000111⟩  |
| 17      | $2^6$ | \|100110⟩  |
| 18      | $2^6$ | \|011001⟩  |
| 19      | $2^6$ | \|110100⟩  |
| 20      | $2^6$ | \|001011⟩  |
| 21      | $2^7$ | \|0101010⟩ |
| 22      | $2^7$ | \|1010101⟩ |
| 23      | $2^7$ | \|1100110⟩ |
| 24      | $2^7$ | \|0011001⟩ |
| 25      | $2^7$ | \|1111000⟩ |
| 26      | $2^7$ | \|0000111⟩ |
| 27      | $2^7$ | \|1001011⟩ |
| 28      | $2^7$ | \|0110100⟩ |
| 29      | $2^7$ | \|1011010⟩ |
| 30      | $2^7$ | \|0100101⟩ |

Для каждого варианта:

1. Реализуйте оракул функцией `build_oracle` из задания 1 (не вручную). Выполните 15 итераций и получите точные $P_{\text{точн}}(k)$ и экспериментальные $P_{\text{эксп}}(k)$ по 1024 shots для $k = 0, \dots, 15$. Постройте графики по группам $N$.

2. Заполните сводную таблицу: вариант, $N$, $|w\rangle$, $k_{\text{теор}} = \lfloor\frac{\pi}{4}\sqrt{N}\rfloor$, $k_{\text{точн}}$ (первый максимум), $k_{\text{эксп}}$, $\max P_{\text{точн}}$.

3. Зависит ли оптимальное число итераций от отмеченного состояния при $M=1$? Опирайтесь на таблицу: сравните разброс $k_{\text{эксп}}$ внутри каждой группы с $N$ и с $k_{\text{точн}}$. Объясните, почему $k_{\text{эксп}}$ может отличаться от $k_{\text{точн}}$, даже когда точный ответ одинаков для всех состояний группы.

4. Для варианта 1 ($N = 2^5$) смоделируйте двухкубитный деполяризующий шум на вентилях CX при $p_2 = 0,005$ и $p_2 = 0,02$. Разложите схему в базис симулятора через `transpile`: MCZ превратится во много вентилей CX. Как шум меняет максимум $P(k)$ и положение оптимума? Объясните, почему глубина схемы важна при шуме.



### Решение Заданий 4-7: Алгоритм Гровера (Вариант 7)


In [1]:
# Решение Заданий 4-7 (Вариант 7)
# N = 2^5 = 32, отмеченное состояние |10010>
import math
import numpy as np

def build_oracle(n, marked_str):
    N = 1 << n
    idx = int(marked_str, 2)
    diag = np.ones(N)
    diag[idx] = -1.0
    return diag

n = 5
N = 1 << n
marked_str = '10010'
idx = int(marked_str, 2)
oracle = build_oracle(n, marked_str)

state = np.full(N, 1.0 / math.sqrt(N))
k_theor = int(math.floor((math.pi / 4.0) * math.sqrt(N)))

print(f'Вариант 7: n = {n}, N = {N}, marked = |{marked_str}>')
print(f'Теоретический оптимум k_теор = floor(pi/4 * sqrt({N})) = {k_theor}\n')
print(' k | Точная вероятность P(k) | Симуляция (1024 shots) | Примечание')
print('---+------------------------+------------------------+------------')

probs = []
for k in range(16):
    prob = state[idx] ** 2
    probs.append(prob)
    rng = np.random.default_rng(42 + k)
    sim_counts = rng.multinomial(1024, state ** 2)
    sim_prob = sim_counts[idx] / 1024.0
    marker = '<-- ТЕОРЕТИЧЕСКИЙ ОПТИМУМ' if k == k_theor else ''
    print(f'{k:2d} |        {prob:.5f}         |        {sim_prob:.5f}         | {marker}')
    
    # Итерация Гровера
    state *= oracle
    mean = np.mean(state)
    state = 2.0 * mean - state

k_max = int(np.argmax(probs))
print(f'\n[+] Итог:')
print(f'    k_теор = {k_theor}')
print(f'    k_точн = {k_max} (P = {probs[k_max]*100:.2f}%)')
assert k_max == k_theor
print('    assert k_max == k_theor подтверждён!')
